# Week 70, Day 3: Review: concurrency

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w070d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def speedup(parallel, workers):
    return 1 / ((1 - parallel) + parallel / workers)

for workers in (2, 4, 16, 1000):
    print(workers, round(speedup(0.9, workers), 2))

In [ ]:
import functools, timeit

@functools.lru_cache(maxsize=None)
def paths(r, c):
    if r == 0 or c == 0:
        return 1
    return paths(r - 1, c) + paths(r, c - 1)

print(paths(16, 16), paths.cache_info().hits, paths.cache_info().misses)

data = list(range(2000))
as_set = set(data)
slow = timeit.timeit(lambda: 1999 in data, number=2000)
fast = timeit.timeit(lambda: 1999 in as_set, number=2000)
print("set is faster:", fast < slow)

In [ ]:
def worker(name, waits):
    for w in waits:
        yield w                    # "await a sleep of w seconds"
    return name

def schedule(tasks):
    now, ready, sleeping, finished = 0, list(tasks.items()), [], []
    while ready or sleeping:
        for name, gen in ready:
            try:
                sleeping.append((now + next(gen), name, gen))
            except StopIteration:
                finished.append((name, now))
        ready = []
        if sleeping:
            sleeping.sort()
            wake, name, gen = sleeping.pop(0)
            now = wake
            ready = [(name, gen)]
    return finished, now

tasks = {"a": worker("a", [2, 1]), "b": worker("b", [1]), "c": worker("c", [3])}
print(schedule(tasks))
print("one after another would take", 2 + 1 + 1 + 3)

In [ ]:
import ctypes

libc = ctypes.CDLL(None)
libc.strlen.argtypes = [ctypes.c_char_p]
libc.strlen.restype = ctypes.c_size_t
print(libc.strlen(b"atlas"))

buf = ctypes.create_string_buffer(b"hello", 10)
libc.memset(buf, ord("*"), 3)
print(buf.value)

> **Key idea.** Measure first (perf_counter, timeit, cProfile, tracemalloc); fix complexity before constants; use sets, dicts, deques, heaps and bisect; cache pure functions; vectorise with NumPy. Amdahl's law limits parallel speed-up by the serial part. Processes give parallelism with separate memory; threads share memory and, under the GIL, help only I/O-bound work; protect shared state with locks or avoid sharing. futures unify pools; asyncio runs coroutines on one thread, switching at await, so never block in a coroutine. Choose by the shape of the work. Extend Python (C API, ctypes, cffi, Cython, PyO3) only after profiling, when nothing simpler works.

### Exercise: How many workers?

Write `speedup(parallel, workers)` (Amdahl's law: `1 / ((1 - parallel) + parallel / workers)`) and `workers_needed(parallel, target)`: the smallest whole number of workers whose speed-up reaches `target`, or `None` if no number of workers can (the speed-up never reaches `1 / (1 - parallel)`; when `parallel` is 1, any target is reachable). Allow for rounding: count a speed-up within `1e-9` of the target as reaching it.

In [ ]:
def speedup(parallel, workers):
    ___

def workers_needed(parallel, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_amdahl():\n    \"Amdahl's law\"\n    assert abs(speedup(0.9, 4) - 3.0769230769) < 1e-6 and speedup(0, 8) == 1\n    assert workers_needed(0.9, 3) == 4 and workers_needed(0.5, 1.5) == 3\n    assert workers_needed(0.9, 10) is None and workers_needed(0.9, 12) is None\n    assert workers_needed(1.0, 50) == 50 and workers_needed(0.75, 1) == 1")

### Exercise: Replaying a race

Two or more threads each run `counter += 1` once, as two steps: **read** (copy the shared counter into the thread's own variable) and **write** (store that variable plus one). Write `replay(schedule)`: `schedule` is a list of thread names, each entry meaning "that thread takes its next step" (its first entry is its read, its second its write). Return `(counter, lost)`: the final counter (starting at 0) and how many increments were lost. Then `safe_schedules(threads)`: of all the orderings of each thread's two steps (every thread name appears twice), how many give no lost updates? (For two threads there are 6 orderings.)

In [ ]:
import itertools

def replay(schedule):
    ___

def safe_schedules(threads):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_race():\n    \"lost updates\"\n    assert replay([\"a\", \"a\", \"b\", \"b\"]) == (2, 0)\n    assert replay([\"a\", \"b\", \"a\", \"b\"]) == (1, 1)\n    assert replay([\"a\", \"b\", \"c\", \"c\", \"b\", \"a\"]) == (1, 2)\n    assert safe_schedules([\"a\", \"b\"]) == 2\n    assert safe_schedules([\"a\", \"b\", \"c\"]) == 6")

### Exercise: A tiny event loop

Coroutines here are generators: each `yield` hands the loop a number of seconds to sleep, and the generator's `return` value is its result. Write `run_loop(coros)`: `coros` is a dict of name to generator. Start them all at time 0, in the dict's order; whenever a task yields `d`, it wakes at `now + d`; the loop always resumes the task that wakes earliest (ties in the order they went to sleep); a task that finishes records its result. Return `(results, finish_times, total)`: dicts of name to result and to finish time, and the time the last one finished.

In [ ]:
import heapq

def run_loop(coros):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_loop():\n    \"an event loop\"\n    def job(waits, result):\n        for w in waits:\n            yield w\n        return result\n    results, finished, total = run_loop({\"a\": job([2, 2], \"A\"), \"b\": job([1], \"B\"), \"c\": job([3], \"C\")})\n    assert results == {\"a\": \"A\", \"b\": \"B\", \"c\": \"C\"}\n    assert finished == {\"a\": 4, \"b\": 1, \"c\": 3} and total == 4\n    order = []\n    def noisy(name, waits):\n        for w in waits:\n            order.append((name, w))\n            yield w\n        return name\n    run_loop({\"x\": noisy(\"x\", [1, 1]), \"y\": noisy(\"y\", [1])})\n    assert order == [(\"x\", 1), (\"y\", 1), (\"x\", 1)]\n    assert run_loop({\"now\": job([], 7)}) == ({\"now\": 7}, {\"now\": 0}, 0) and run_loop({}) == ({}, {}, 0)")

### Exercise: Choosing a model

Write `choose(task)`. A task is a dict with `"compute"` (seconds of Python computing per item), `"wait"` (seconds of waiting on the network or disk per item) and `"items"`. Return `"sequential"` if the whole job would take under a second done one at a time; otherwise `"processes"` if computing is at least half of each item's time; otherwise `"asyncio"` if there are more than 100 items; otherwise `"threads"`. Then `estimate(task, model, workers=4)`: the time in seconds the job would take: sequential is `items * (compute + wait)`; threads and asyncio overlap all the waiting but compute one at a time (`items * compute + wait`); processes split everything over `workers` (`items * (compute + wait) / workers`).

In [ ]:
def choose(task):
    ___

def estimate(task, model, workers=4):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_choose():\n    \"the shape of the work\"\n    assert choose({\"compute\": 0.001, \"wait\": 0.001, \"items\": 10}) == \"sequential\"\n    assert choose({\"compute\": 2, \"wait\": 0.5, \"items\": 8}) == \"processes\"\n    assert choose({\"compute\": 0.001, \"wait\": 0.2, \"items\": 5000}) == \"asyncio\"\n    assert choose({\"compute\": 0.01, \"wait\": 0.5, \"items\": 20}) == \"threads\"\n    t = {\"compute\": 0.01, \"wait\": 0.5, \"items\": 20}\n    assert abs(estimate(t, \"sequential\") - 10.2) < 1e-9 and abs(estimate(t, \"threads\") - 0.7) < 1e-9\n    assert abs(estimate(t, \"processes\", workers=5) - 2.04) < 1e-9 and estimate(t, \"asyncio\") == estimate(t, \"threads\")")

### Exercise: Sorting in C

Write `c_sorted(values, reverse=False)`: copy the integers into a `ctypes` array of `c_int`, sort it with the C library's `qsort` (from `ctypes.CDLL(None)`) using a `CFUNCTYPE` comparison callback, and return a Python list. Don't call `sorted` or `.sort`. Append one entry to the module-level list `CALLS` each time your comparison function is called, so you can see C calling Python.

In [ ]:
import ctypes

CALLS = []

def c_sorted(values, reverse=False):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_c_sorted():\n    \"qsort with a callback\"\n    CALLS.clear()\n    assert c_sorted([5, 3, 9, 1, 7]) == [1, 3, 5, 7, 9] and len(CALLS) > 0\n    assert c_sorted([5, 3, 9, 1, 7], reverse=True) == [9, 7, 5, 3, 1]\n    assert c_sorted([]) == [] and c_sorted([-2, 2, 0]) == [-2, 0, 2]\n    assert \"sorted(\" not in source.replace(\"c_sorted(\", \"\") and \".sort(\" not in source")

### Bug hunt: A lock left held

`record` acquires the lock by hand and releases it only on success, so the failed call leaves it locked and the next call finds it busy: this prints `['ok', 'busy']`. Use `with lock:` so the lock is always released and this prints `['ok', 'ok']`.

In [ ]:
import threading

lock = threading.Lock()
log = []

def record(value):
    if not lock.acquire(blocking=False):
        log.append("busy")
        return
    if value < 0:
        raise ValueError(value)
    log.append("ok")
    lock.release()

record(1)
try:
    record(-1)
except ValueError:
    pass
record(2)
print(log)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"always release\"\n    expect_output(\"['ok', 'ok']\")")

### Bug hunt: A cache keyed on half the arguments

The memo uses only `base` as its key, so `power(2, 5)` returns the answer it saved for `power(2, 3)`: this prints `8 8`. Key the cache on both arguments so this prints `8 32`.

In [ ]:
memo = {}

def power(base, exp):
    if base not in memo:
        memo[base] = base ** exp
    return memo[base]

print(power(2, 3), power(2, 5))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a complete key\"\n    expect_output(\"8 32\")")

### Bug hunt: Membership in a list, inside a loop

`common` checks `x in b` with `b` a list, scanning it for every item: the answer is right but the work is quadratic, and the check below counts the comparisons. Convert `b` to a set once, before the loop, so the count is small and this prints `[3, 4] True`.

In [ ]:
class Counting(list):
    checks = 0
    def __contains__(self, item):
        Counting.checks += len(self)
        return super().__contains__(item)

def common(a, b):
    return [x for x in a if x in b]

b = Counting(range(3, 1000))
print(common([1, 2, 3, 4], b), Counting.checks < 100)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a set for membership\"\n    expect_output(\"[3, 4] True\")")

### Puzzle: A thread-safe counter

Arrange the lines of a counter whose increments can't be lost, even with many threads. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # round(1 / (0.5 + 0.5 / 2), 2)
q = ___   # True or False: two threads run Python bytecode at the same instant in standard CPython
r = ___   # the best model for 10,000 slow network requests: "threads", "processes" or "asyncio"
s = ___   # who gets control back at an await: "the event loop" or "the operating system"
t = ___   # True or False: ctypes needs a C compiler

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 1.33 and q is False and r == \"asyncio\", \"p, q and r\"\n    assert s == \"the event loop\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a limited, de-duplicating fetcher

Simulate a fetcher that combines a cache, request sharing and a concurrency limit. `simulate(requests, cost, limit)`: `requests` is a list of `(time, key)` in time order; fetching `key` takes `cost[key]` seconds; at most `limit` fetches run at once, and waiting fetches start in the order they were queued as soon as a slot frees. For each request: if `key` has already **finished** being fetched (at or before the request time), it's a **hit** and finishes at its request time; if it's running or queued, it's **shared** and finishes when that fetch does; otherwise it starts a new fetch (or queues one). Return `(finish_times, stats)`: a list of finish times, one per request, and a dict of counts `{"hits": ..., "shared": ..., "fetches": ...}`. When fetches finish at the same time, free their slots in the order they started.

In [ ]:
import heapq
from collections import deque

def simulate(requests, cost, limit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fetcher():\n    \"caching, sharing and a limit together\"\n    cost = {\"a\": 3, \"b\": 2, \"c\": 1, \"d\": 4}\n    times, stats = simulate([(0, \"a\"), (0, \"b\"), (0, \"c\"), (1, \"a\"), (3, \"a\"), (4, \"c\")], cost, 2)\n    assert times == [3, 2, 3, 3, 3, 4]\n    assert stats == {\"hits\": 2, \"shared\": 1, \"fetches\": 3}\n    times2, stats2 = simulate([(0, \"d\"), (0, \"a\"), (0, \"b\"), (0, \"b\"), (10, \"d\")], cost, 1)\n    assert times2 == [4, 7, 9, 9, 10] and stats2 == {\"hits\": 1, \"shared\": 1, \"fetches\": 3}\n    assert simulate([], cost, 3) == ([], {\"hits\": 0, \"shared\": 0, \"fetches\": 0})")

## Warm-up

Even with 90% of the work parallel, a thousand workers give less than a tenfold speed-up: the serial 10% never shrinks. That's Amdahl's law (Week 66 Day 1), and it's why you measure what's serial before buying more cores.

## Performance

- **Measure first** (Week 65 Day 1): `time.perf_counter` for one run, `timeit` for small snippets (it repeats and turns off garbage collection; take the minimum), `cProfile` and `pstats` to find **where** time goes, `tracemalloc` for memory. Optimise the hot spot, not the code you suspect.
- **Algorithms beat constants** (Day 2): an O(n²) loop hidden in `x in list` inside a loop, `list.pop(0)` or repeated string concatenation dwarfs any micro-tuning. Fix the complexity first, then hoist repeated work out of loops.
- **Fast structures** (Day 3): `set` and `dict` for membership, `collections.deque` for queues, `heapq` for the best k, `bisect` for sorted data, `Counter` and `defaultdict` for grouping, `array` and `__slots__` to save memory.
- **Caching** (Day 4): `functools.cache` and `lru_cache` for pure functions with hashable arguments; think about invalidation, size limits and expiry when results can go stale.
- **Vectorisation** (Day 5): NumPy runs whole-array operations in C. Watch for views versus copies, fixed-size integer overflow, and floating-point comparisons.

## Concurrency

- **Processes and threads** (Week 66 Day 1): processes have separate memory (safe, but data must be copied between them); threads share memory (cheap to talk, easy to corrupt). **Concurrency** is dealing with many things at once; **parallelism** is doing them at the same instant.
- **The GIL** (Day 3): in standard CPython only one thread runs Python bytecode at a time, so threads help **I/O-bound** work (the GIL is released while waiting) but not **CPU-bound** pure-Python work. Free-threaded builds (3.13 on) remove it, at some cost.
- **Races and locks** (Day 4): `counter += 1` is a read, an add and a write, and a thread switch between them loses updates. Protect shared state with `threading.Lock` (always through `with`), or avoid sharing: give each worker its own data and combine results, or pass messages through a `queue.Queue`. Lock in a consistent order to avoid deadlock.
- **`concurrent.futures`** (Day 5): `ThreadPoolExecutor` and `ProcessPoolExecutor` share one interface: `submit` returns a **future**; `map`, `as_completed` and `wait` collect results; exceptions come back when you call `result()`.
- **`multiprocessing`** (Week 67 Day 1): real parallelism for CPU work; guard the start with `if __name__ == "__main__":`, keep arguments picklable, and send big chunks rather than many tiny tasks.
- **`asyncio`** (Days 2 to 4): coroutines (`async def`) pause at `await`, handing control back to a single-threaded **event loop**, which runs whichever task is ready. `gather`, `create_task`, `wait_for`, `TaskGroup`, `Semaphore` and `Queue` coordinate them. A blocking call (`time.sleep`, heavy computing) freezes **every** task; use `await asyncio.sleep` and `asyncio.to_thread`.
- **Choosing** (Day 5): CPU-bound, processes; a few blocking I/O calls or blocking libraries, threads; thousands of network waits, asyncio. Measure, because the shape of the work decides.

The three tasks wait at the same time, so the whole run takes as long as the longest chain of waits (3), not the total (7). That's all an event loop does: it never runs two things at the same instant, it just never sits idle while something could run.

## Extending Python

- **The C API** (Week 68 Day 1): extension modules are C code built against `Python.h`, managing reference counts by hand (`Py_INCREF`, `Py_DECREF`); the limited API and stable ABI let one build work across versions.
- **`ctypes`** (Day 2) calls functions in existing shared libraries from pure Python: set `argtypes` and `restype`, pass `bytes` for `char *`, use `byref`, arrays, `Structure` and `CFUNCTYPE` callbacks. Mistakes crash rather than raise. **`cffi`** reads C declarations directly.
- **Cython** (Day 3) compiles Python-like code with optional C types to a C extension; typed loops get large speed-ups, and `nogil` blocks run in parallel.
- **PyO3** (Day 4) writes extension modules in Rust, with `maturin` to build them; Rust's ownership rules prevent whole classes of memory bugs.
- **When to extend** (Day 5): only after profiling, when the hot spot is a tight loop no library already handles. Try a better algorithm, the standard library, NumPy or an existing package first; count the cost of a build step, of platform wheels and of debugging across languages.

## Classic mistakes

- **Optimising without measuring**, or timing the setup together with the work.
- **Threads for CPU-bound pure Python** under the GIL: no faster, sometimes slower.
- **Unprotected read-modify-write** on shared state, or a lock acquired without `with`, so an exception leaves it held.
- **Blocking inside a coroutine** (`time.sleep`, a long loop), or calling a coroutine function without awaiting it, so it never runs.
- **Caching with unhashable or incomplete keys**, or caching results that go stale.
- **Extending before trying a better algorithm**, a library or NumPy.

## Practice

## On your own computer

The real versions of today's simulations, for a terminal with standard CPython:

```python
import asyncio, time
from concurrent.futures import ProcessPoolExecutor

def crunch(n):
    return sum(i * i for i in range(n))

async def fetch(name, seconds):
    await asyncio.sleep(seconds)
    return name

async def main():
    start = time.perf_counter()
    results = await asyncio.gather(fetch("a", 1), fetch("b", 1), fetch("c", 1))
    print(results, round(time.perf_counter() - start, 1), "seconds")   # about 1, not 3

if __name__ == "__main__":
    asyncio.run(main())
    with ProcessPoolExecutor() as pool:
        print(sum(pool.map(crunch, [2_000_000] * 8)))
```

Time the `ProcessPoolExecutor` part against a plain loop and against a `ThreadPoolExecutor`: on a standard build the threads won't beat the loop, and the processes will, by roughly your number of cores.

## Quiz

## Daily challenge

## Before you go

Look at your list from the three review days and pick the three weakest ideas: revise those tomorrow after the practice exam. Tomorrow: a practice exam in Friday's format.